In [33]:
%load_ext ipympl
%matplotlib widget

import numpy as np
import importlib, game_3dutils, game_costs,neos_path_game, ukf_estimator,game_viz
from pathlib import Path
import shutil
import matplotlib.pyplot as plt

importlib.reload(game_3dutils)
importlib.reload(game_costs)
importlib.reload(neos_path_game)
importlib.reload(ukf_estimator)
importlib.reload(game_viz)


from game_3dutils    import run_rhc_and_collect_frames_3d, run_rhc_and_collect_frames_3d_N
from game_costs import build_game_costs
from game_viz import animate_rollout_3d, interactive_rollout_3d



The ipympl module is not an IPython extension.


In [36]:
# -----------------------------------------------------------------------------
# Config and notation
# -----------------------------------------------------------------------------
# State x = [px, py, vx, vy]ᵀ: position (px, py) and velocity (vx, vy)
# Control u = [ax, ay]ᵀ: acceleration inputs
# Horizon length T: number of state samples (there are T-1 control samples)
# Players: N=2 (player 1, player 2). Trajectories for each player i are packed into τᵢ.
# τ = [τ₁; τ₂]: concatenation of both players’ trajectories.
# θ (“theta”): parameter vector containing the stacked initial states for both players.
#
# We form a “variational inequality” style equilibrium by enforcing stationarity of each
# player’s Lagrangian w.r.t. its own τᵢ, plus shared dynamics/IC constraints g̃(τ,θ)=0
# and shared inequalities h̃(τ,θ) ≥ 0 (arena, bounds, separation, obstacles).
# -----------------------------------------------------------------------------


CONFIG = {
    # --- scenario / time ---
    "solver_kind": "path",
    # PATH solver settings
    "pathampl": "/Users/gussantaella/Documents/UTAustin/Research/Code/Research_Repo/path_5/ampl/pathampl",
    "path_eps_R": 1e-3,

    # --- sim grid ---
    "setting":  "chase_escape_tail", #chase_escape_tail, rendezvous_track, generic_n
    "N": 2,
    "D":        3,        # workspace dimension
    "T":        5,        # horizon length (steps)
    "dt":       0.1,      # step size (s)
    "sim_time": 45.0,     # rollout duration (s)

    # --- dynamics ---
    "dynamics": "hcw",    # "hcw" or "double"
    "hcw": {"mu": 3.986004418e14, "r0": 6_371_000.0 + 400_000.0},

    # --- initial states: [px,py,pz,vx,vy,vz] ---
    # Initial velocity sets initial boresight; keep vx,vy,vz ≠ 0 if you care about direction.
    "x0": np.array([
        [ -2.0,  0.0, 0.0,  -0.02, 0.00,  0.000],  # agent 1
        [-15.0,  0.0, 0.0,   0.00, 0.00, -2.000],  # agent 2
    ], dtype=float),

    # --- workspace / constraints ---
    "arena": {"type": "sphere", "cx": 0.0, "cy": 0.0, "cz": 0.0, "r": 20.0},
    "vmax":   10.0,   # |v| bound used to form x bounds
    "umax":   10.0,   # |a| bound used to form u bounds
    "sep_min": 6,   # pairwise separation (m)
    "spheres": [],    # optional keep-outs: [{"cx":..,"cy":..,"cz":..,"r":..}, ...]

    # --- field of view (pinhole; cone fields kept for legacy code paths) ---
    "fov": {
        "enabled": True,
        "type": "pinhole",
        "agent": 1,                 # camera mounted on agent 1 (set 2 to flip)
        "color": "C1",
        "alpha": 0.15,
        "range": 3.0,               # legacy (for cone)
        "hfov_deg": 60.0,           # legacy (for cone)
        "min_speed_for_axis": 1e-3  # fallback if attitude config not set
    },

    # --- camera intrinsics / frustum (must match att['align']) ---
    "camera": {
        "W": 1280, "H": 720,
        "fx": 800.0, "fy": 800.0,
        "cx": 640.0, "cy": 360.0,
        "near": 0.05,
        "far":  1.5,
        "align": "x",               # match att['align'] (x-forward boresight)
    },

    # --- attitude (boresight from velocity; optional roll in state) ---
    "att": {
        "enabled": True,
        "mode": "hold",             # "hold" | "rate" | "track"
        "align": "x",               # body x̂ is boresight (matches camera.align)
        "up": [0.0, 0.0, 1.0],      # world up vector
        "min_speed_for_axis": 1e-4, # hold boresight if |v| < this

        "init": {
            "phi0": [np.pi/4, 0.0],     # per-agent roll seeds (rad)
            # Let code seed boresight from initial speed.
            # Keep these disabled to avoid overrides:
            # "look_at_other": False,
            # "axis0": [[...],[...]],
        },

        "stabilize_up": False,      # if True, tries to keep ẑ_b ~ world_up (can flip near poles)
        "roll_enabled": True,       # if False, ignore φ (treat as 0)
    },

    # --- UKF / estimation (bearing-only CV filter) ---
    "est": {
        "enabled": True,                 # run the KF
        "who": "both",                   # '1->2', '2->1', or 'both'
        "every": 1,                      # take a bearing each step (good for debugging)
        "meas_std_deg": (0.3, 0.3),      # az/el std dev (deg)
        "P0_diag": [25,25,25, 1,1,1],    # diag for 6x6 P0
        "Q_diag" : [1e-4,1e-4,1e-4, 1e-3,1e-3,1e-3],  # process noise diag
    },

    # --- visualization (animate_rollout_3d) ---
    "viz": {
        "only_est": False,          # show ONLY UKF dotted tracks (hide plans/execution)
        "show_est": True,          # draw the estimate lines
        "show_meas": True,         # draw current az/el ray from observer
        'filter': 'ekf',   # 'ukf' or 'ekf'
        "meas_len": 2.0,           # length of the ray in world units
        # triads (kept here so you can tweak without touching code)
        "triad_len": [0.6, 0.4, 0.4],
        "triad_colors": ["tab:red","tab:green","tab:blue"],
        "triad_labels": ["x_b (boresight)","y_b","z_b"],
    },
}

CONFIG["viz"]["axis_scale"] = 1e2
CONFIG["viz"]["axis_unit"] = "m"
CONFIG["viz"]["axis_label_only"] = True   # <- leaves numbers as-is

CONFIG["est"] = {
    "enabled": True,
    "who": "both",   # or "all"
    "every": 1,
    "meas_std_deg": (0.3, 0.3),
}


In [37]:
if __name__ == "__main__":

    # 2) Generate rollout DATA
    # rollout_OG = run_rhc_and_collect_frames_3d(cfg=CONFIG) 
    # rollout = run_rhc_and_collect_frames_3d_N(cfg=CONFIG)
    rollout = run_rhc_and_collect_frames_3d_N(cfg=CONFIG)


    # rollout = run_rhc_and_collect_frames_3d(cfg=CONFIG)
    # raise("Rollout finished")

    # 3) Make a GIF or MP4 directly from the rollout (pick one)
    # GIF:
    animate_rollout_3d(rollout, save_path="traj_3D.gif", fps=20, cfg=CONFIG, show_fov=True, show_axes=True)

    import imageio.v3 as iio
    gif_frames = iio.imread("traj_3D.gif")
    iio.imwrite("traj_3D.mp4", gif_frames, fps=20, codec="h264")
    print("Saved animation to traj_3D.mp4")

    # 4) INTERACTIVE viewer (Jupyter/Colab)
    interactive_rollout_3d(rollout, CONFIG, show_fov=True, show_axes=True)
    plt.show()  # ensure it displays



#Deletes locally stashed Python cache.
for p in Path('.').rglob('__pycache__'):
    shutil.rmtree(p, ignore_errors=True)
for f in Path('.').rglob('*.pyc'):
    f.unlink(missing_ok=True)


ValueError: not enough values to unpack (expected 2, got 1)

In [47]:
import numpy as np

def _I3():
    return np.eye(3)

def _att_exec_stub(n):
    I = _I3()
    return [{"R": I, "phi": 0.0} for _ in range(n)]

def _att_plan_stub(nsteps, T):
    I = _I3()
    return [[{"R": I, "phi": 0.0} for _ in range(T)] for _ in range(nsteps)]

def as_legacy_2p(rollout, T_hint=None):
    # Only adapt when N=2-style data is present
    if "exec_hist" not in rollout or len(rollout["exec_hist"]) != 2:
        return rollout

    exec_hist = rollout["exec_hist"]
    plan_hist = rollout.get("plan_hist", [[], []])

    n_exec  = len(exec_hist[0])
    n_steps = len(plan_hist[0]) if plan_hist and plan_hist[0] else 0
    T       = len(plan_hist[0][0]) if n_steps else (T_hint or 1)

    I = _I3()
    exec_att1 = [{"R": I, "phi": 0.0} for _ in range(n_exec)]
    exec_att2 = [{"R": I, "phi": 0.0} for _ in range(n_exec)]
    plan_att1 = [[{"R": I, "phi": 0.0} for _ in range(T)] for _ in range(n_steps)]
    plan_att2 = [[{"R": I, "phi": 0.0} for _ in range(T)] for _ in range(n_steps)]

    fov_axis_hist = [np.array([1.0, 0.0, 0.0])] * n_exec
    fov_seen_mask = [False] * n_exec

    # Build legacy 2p structure expected by game_viz
    return {
        **rollout,
        "plan_hist1": plan_hist[0],
        "plan_hist2": plan_hist[1],
        "exec1_xyz":  exec_hist[0],
        "exec2_xyz":  exec_hist[1],
        "plan_att1":  plan_att1,
        "plan_att2":  plan_att2,
        "exec_att1":  exec_att1,
        "exec_att2":  exec_att2,
        "phi_hist1":  [0.0] * n_exec,
        "phi_hist2":  [0.0] * n_exec,
        "fov_axis_hist": fov_axis_hist,
        "fov_seen_mask": fov_seen_mask,
    }


In [18]:
rollout_2p = as_legacy_2p(rollout, T_hint=CONFIG.get("T"))

# rollout = run_rhc_and_collect_frames_3d(cfg=CONFIG)
# raise("Rollout finished")

# 3) Make a GIF or MP4 directly from the rollout (pick one)
# GIF:
animate_rollout_3d(rollout_2p, save_path="traj_3D.gif", fps=20, cfg=CONFIG, show_fov=True, show_axes=True)


Saved 3D animation to traj_3D.gif
